# Water Audit Queue | How should a portfolio allocate limited reviews?

CE 663 · Integrated Water Management

This study asks how many properties that report elevated water use next year can be surfaced with a fixed number of review slots. The evaluation compares intensity, deviation from peers, volume above the benchmark, and a simple risk model. It also measures the trade-off between finding more elevated properties and capturing more gallons above the threshold.

Run with Python 3, NumPy and pandas. The included CSV files are read from the package’s data folder. Open the notebook from the notebooks folder or run it from the package root. The saved outputs contain the original results. docs/QUICKSTART.md explains the command-line runner.

[NYC disclosure data](https://data.cityofnewyork.us/Environment/NYC-Building-Energy-and-Water-Data-Disclosure-for-/5zyy-y8am/about_data)

# Portfolio audit study protocol

This is a retrospective extension of the previously scored 2024 benchmark. Freeze this protocol before inspecting the new ranking outcomes. The research question is whether a ranking that uses a property's peer-adjusted water use or estimated above-benchmark volume can allocate scarce reviews more effectively than sorting by reported intensity.

Use the existing frozen property groups and record quality screen. Estimate the peer benchmark with the existing 2022 area-and-occupancy regression fitted on training groups only. Define elevated use as an observed-to-expected intensity ratio above the 90th percentile of 2022 records in the validation groups. Freeze this threshold for every year. Link consecutive years by exact property ID after screening both years. Train a logistic risk score on 2022-to-2023 transitions in training groups, using log observed-to-expected ratio, log floor area and occupancy from the earlier year. Compare four policies on 2022-to-2023 validation transitions in the validation groups: raw intensity, observed-to-expected ratio, current positive gallons above expected, and predicted probability of an elevated following year. Select the demo default by validation precision at 10% capacity, breaking ties in that order. Keep the models and threshold fixed and evaluate every policy on 2023-to-2024 transitions in the reserved test groups.

At 5%, 10% and 20% review capacity, round the available seats up, break score ties by property ID, and report the share of selected properties with elevated next-year use (precision), the share of next-year elevated properties selected (recall), and the share of total next-year gallons above the elevated-use cutoff captured by the queue. Compare 10% precision against the raw-intensity policy with paired 95% group-bootstrap intervals (1,000 resamples, seed 663). Report the share of screened 2023 candidates with eligible 2024 follow-up, and compare their starting intensity and high-use rates with those missing follow-up. As a sensitivity check, recompute 10% precision with 85th- and 95th-percentile outcome definitions, without refitting or reranking. Annual disclosures contain no confirmed leaks, water savings or audit outcomes.

## Load data and reuse the frozen property groups

Group identifiers come from the earlier benchmark study and the same annual record screen applies.

In [1]:
from hashlib import sha256
from pathlib import Path
from urllib.parse import urlencode
import io
import re

import numpy as np
import pandas as pd

fields = [
    'report_year', 'property_id', 'parent_property_id', 'nyc_borough_block_and_lot',
    'nyc_building_identification', 'primary_property_type_self', 'primary_property_type',
    'number_of_buildings', 'occupancy', 'property_gfa_self_reported',
    'property_gfa_calculated_1', 'water_use_all_water_sources', 'metered_areas_water',
    'estimated_values_water', 'alert_water_meter_has_less',
]

def read_extract(expected_years, local_names):
    year_list = ','.join(map(str, expected_years))
    query = {
        '$select': ','.join(fields),
        '$where': f"report_year in ({year_list}) and primary_property_type_self = 'Multifamily Housing'",
        '$limit': 100000,
        '$order': 'report_year,property_id,:id',
    }
    url = 'https://data.cityofnewyork.us/resource/5zyy-y8am.csv?' + urlencode(query)
    expected = {str(year) for year in expected_years}
    for folder in (Path.cwd() / 'data', Path.cwd().parent / 'data'):
        for name in local_names:
            path = folder / name
            if not path.is_file():
                continue
            data = path.read_bytes()
            try:
                frame = pd.read_csv(io.BytesIO(data), dtype='string', usecols=fields)
            except ValueError:
                continue
            if set(frame.report_year.unique()) == expected and frame.primary_property_type_self.eq('Multifamily Housing').all():
                return frame, sha256(data).hexdigest()
    raise FileNotFoundError(f'Place the included CSV extracts in the package data folder. Source query {url}')


early, early_sha = read_extract([2022, 2023], ['nyc_multifamily_2022_2023.csv'])
future, future_sha = read_extract([2024], ['nyc_multifamily_2024.csv'])
raw = pd.concat([early, future], ignore_index=True)
print('2022–23 extract SHA-256:', early_sha)
print('2024 extract SHA-256:', future_sha)
print('Raw multifamily records by year:', raw.report_year.value_counts().sort_index().to_dict())

2022–23 extract SHA-256: a2ff47070d532c78e9d2116cb73682246bd8c8b6c6e76308b741bea5686381cf
2024 extract SHA-256: b7e9f2348d6656ca66057f0e28f4efe5aec1d207f4498105f56c3aa21cd36a35
Raw multifamily records by year: {'2022': 19732, '2023': 22219, '2024': 26745}


In [2]:
def property_keys(row):
    keys = set()
    for value in (row.property_id, row.parent_property_id):
        if pd.notna(value) and re.fullmatch(r'[1-9]\d*', value.strip()):
            keys.add(('property', value.strip()))
    if pd.notna(row.nyc_building_identification):
        for number in re.findall(r'\b[1-5]\d{6}\b', row.nyc_building_identification):
            if number[1:] != '000000':
                keys.add(('building', number))
    if pd.notna(row.nyc_borough_block_and_lot):
        lots = row.nyc_borough_block_and_lot
        for borough, block, lot in re.findall(r'\b([1-5])\s*-\s*(\d{1,5})\s*-\s*(\d{1,4})\b', lots):
            if int(block) and int(lot):
                keys.add(('lot', f'{borough}{int(block):05d}{int(lot):04d}'))
        for number in re.findall(r'\b[1-5]\d{9}\b', lots):
            if int(number[1:6]) and int(number[6:]):
                keys.add(('lot', number))
    return keys


def root(parents, item):
    while parents[item] != item:
        parents[item] = parents[parents[item]]
        item = parents[item]
    return item


def bucket(salt, group):
    return int.from_bytes(sha256(f'{salt}:{group}'.encode()).digest(), 'big') % 100


metadata = raw[['property_id', 'parent_property_id', 'nyc_building_identification', 'nyc_borough_block_and_lot']]
metadata = metadata.loc[metadata.property_id.str.fullmatch(r'[1-9]\d*', na=False)].drop_duplicates()
parents = {int(pid): int(pid) for pid in metadata.property_id.unique()}
key_owners = {}
has_site_key = set()
for row in metadata.itertuples(index=False):
    pid = int(row.property_id)
    keys = property_keys(row)
    if any(kind in {'building', 'lot'} for kind, _ in keys):
        has_site_key.add(row.property_id)
    for key in keys:
        owner = key_owners.setdefault(key, pid)
        left, right = sorted((root(parents, pid), root(parents, owner)))
        parents[right] = left

group_by_property = {str(pid): root(parents, pid) for pid in parents}
group_ids = sorted(set(group_by_property.values()))
split_by_group = {}
for group in group_ids:
    if bucket('ce663-water-test-v1', group) < 20:
        split_by_group[group] = 'test'
    elif bucket('ce663-water-val-v1', group) < 25:
        split_by_group[group] = 'validation'
    else:
        split_by_group[group] = 'train'

assert all(group <= int(pid) and group_by_property[str(group)] == group
           for pid, group in group_by_property.items())
split_manifest = '\n'.join(f'{pid},{group},{split_by_group[group]}'
                           for pid, group in sorted(group_by_property.items(), key=lambda item: int(item[0])))
print('Frozen split SHA-256:', sha256(split_manifest.encode()).hexdigest())
print('Groups across raw identifier metadata:', pd.Series(split_by_group).value_counts().to_dict())

Frozen split SHA-256: f19cf84a59562a98d3ccda59585afee72aad3d2803603ebe8724e52eb55ca693
Groups across raw identifier metadata: {'train': 11412, 'test': 3756, 'validation': 3678}


In [3]:
def screen(records):
    unique = records.drop_duplicates().copy()
    identified = unique.property_id.isin(group_by_property)
    conflicts = identified & unique.duplicated(['report_year', 'property_id'], keep=False)
    frame = unique.loc[identified & ~conflicts].copy()
    numeric = ['occupancy', 'property_gfa_self_reported', 'water_use_all_water_sources']
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    eligible = (
        frame.occupancy.gt(0) & frame.occupancy.le(100)
        & frame.property_gfa_self_reported.gt(0) & frame.water_use_all_water_sources.gt(0)
        & np.isfinite(frame.property_gfa_self_reported) & np.isfinite(frame.water_use_all_water_sources)
        & frame.metered_areas_water.eq('Whole Property') & frame.estimated_values_water.eq('No')
        & frame.alert_water_meter_has_less.eq('Ok')
    )
    frame = frame.loc[eligible].copy()
    frame['group'] = frame.property_id.map(group_by_property)
    frame['split'] = frame.group.map(split_by_group)
    frame['wui'] = 1000 * frame.water_use_all_water_sources / frame.property_gfa_self_reported
    return frame, len(records) - len(unique), int(conflicts.sum())


history, early_repeats, early_conflicts = screen(early)
train = history.loc[history.report_year.eq('2022') & history.split.eq('train')].copy()
validation = history.loc[history.report_year.eq('2023') & history.split.eq('validation')].copy()
final_fit = (history.loc[history.split.ne('test')].sort_values('report_year')
             .drop_duplicates('property_id', keep='last').copy())
assert set(train.group).isdisjoint(validation.group)
print('2022–23 exact repeats collapsed:', early_repeats)
print('2022–23 conflicting property-year rows excluded:', early_conflicts)
for name, frame in [('2022 initial train', train), ('2023 validation', validation), ('2022–23 final fit', final_fit)]:
    print(f'{name}: {len(frame):,} properties across {frame.group.nunique():,} groups')

2022–23 exact repeats collapsed: 409
2022–23 conflicting property-year rows excluded: 368
2022 initial train: 8,430 properties across 7,823 groups
2023 validation: 2,549 properties across 2,325 groups
2022–23 final fit: 13,054 properties across 11,246 groups


## Define elevated use and fit the ranking model

A fixed peer benchmark is estimated from 2022 training records. A single relative-use threshold is calibrated with separate 2022 validation groups. All features describe the candidate in the earlier year.

In [4]:
labels = ['Overall median', 'Size-band median', 'Area + occupancy regression']


def features(frame):
    area = frame.property_gfa_self_reported.to_numpy(dtype=float)
    occupancy = frame.occupancy.to_numpy(dtype=float)
    return np.column_stack([np.ones(len(frame)), np.log(area), occupancy / 100])


def fit(frame):
    area = frame.property_gfa_self_reported.to_numpy(dtype=float)
    intensity = frame.wui.to_numpy(dtype=float)
    cuts = np.quantile(area, [0.25, 0.50, 0.75])
    bands = np.searchsorted(cuts, area, side='left')
    medians = [float(np.median(intensity[bands == band])) for band in range(4)]
    coefficients = np.linalg.lstsq(features(frame), np.log(intensity), rcond=None)[0]
    return {
        'overall': float(np.median(intensity)), 'cuts': cuts.tolist(),
        'bandMedians': medians, 'coefficients': coefficients.tolist(),
        'areaReference': np.quantile(area, [0.01, 0.99]).tolist(),
    }


def predict(model, frame):
    areas = frame.property_gfa_self_reported.to_numpy(dtype=float)
    return np.column_stack([
        np.full(len(frame), model['overall']),
        np.asarray(model['bandMedians'])[np.searchsorted(model['cuts'], areas, side='left')],
        np.exp(features(frame) @ model['coefficients']),
    ])


peer = fit(train)
calibration = history.loc[history.report_year.eq('2022') & history.split.eq('validation')]
calibration_ratio = calibration.wui.to_numpy(dtype=float) / predict(peer, calibration)[:, 2]
alternative_thresholds = np.quantile(calibration_ratio, [.85, .90, .95])
threshold = float(alternative_thresholds[1])

policies = ['Reported intensity', 'Peer-adjusted ratio', 'Gallons above expected', 'Predicted risk']


def candidate_features(frame):
    area = frame.property_gfa_self_reported.to_numpy(dtype=float)
    occupancy = frame.occupancy.to_numpy(dtype=float)
    expected = predict(peer, frame)[:, 2]
    intensity = frame.wui.to_numpy(dtype=float)
    ratio = intensity / expected
    return pd.DataFrame({
        'property_id': frame.property_id.to_numpy(), 'group': frame.group.to_numpy(),
        'area': area, 'occupancy': occupancy, 'gallons': frame.water_use_all_water_sources.to_numpy(dtype=float) * 1000,
        'intensity': intensity, 'expected': expected, 'ratio': ratio,
        'above_expected': np.maximum(intensity - expected, 0) * area,
    })


def transitions(first, second):
    current = candidate_features(first)
    following = candidate_features(second)[['property_id', 'group', 'area', 'ratio', 'intensity', 'expected']]
    return current.merge(following, on=['property_id', 'group'], suffixes=('', '_next'), validate='one_to_one')


def risk_features(frame):
    return np.column_stack([np.log(frame.ratio), np.log(frame.area), frame.occupancy / 100])


def sigmoid(value):
    return 1 / (1 + np.exp(-np.clip(value, -50, 50)))


training_pairs = transitions(train, history.loc[history.report_year.eq('2023') & history.split.eq('train')])
validation_pairs = transitions(calibration, validation)
train_x = risk_features(training_pairs)
center, scale = train_x.mean(axis=0), train_x.std(axis=0)
design = np.column_stack([np.ones(len(train_x)), (train_x - center) / scale])
target = training_pairs.ratio_next.to_numpy() > threshold
weights = np.zeros(design.shape[1])
penalty = np.diag([0., 1., 1., 1.])
for _ in range(50):
    probability = sigmoid(design @ weights)
    gradient = design.T @ (probability - target) + penalty @ weights
    curvature = design.T @ ((probability * (1 - probability))[:, None] * design) + penalty
    step = np.linalg.solve(curvature, gradient)
    weights -= step
    if np.max(np.abs(step)) < 1e-8:
        break
else:
    raise RuntimeError('Risk model did not converge')


def rank_scores(frame):
    risk = sigmoid(weights[0] + ((risk_features(frame) - center) / scale) @ weights[1:])
    return np.column_stack([frame.intensity, frame.ratio, frame.above_expected, risk])


print(f'2022 calibration properties: {len(calibration):,}')
print(f'Elevated-use cutoff: {threshold:.3f}× expected intensity')
print(f'Risk model training transitions: {len(training_pairs):,} in {training_pairs.group.nunique():,} groups')
print(f'Validation transitions: {len(validation_pairs):,} in {validation_pairs.group.nunique():,} groups')

2022 calibration properties: 2,687
Elevated-use cutoff: 2.254× expected intensity
Risk model training transitions: 6,581 in 6,270 groups
Validation transitions: 2,094 in 2,001 groups


## Freeze the review policy before evaluating the 2024 outcome

Precision counts the fraction of available review slots that find an elevated report. Recall counts the fraction of all elevated reports found. Volume capture is the share of future gallons above the fixed cutoff, using the following year's area and expected intensity. The estimate describes reported volume above a statistical threshold.

In [5]:
from math import ceil

capacities = (.05, .10, .20)


def ordered_rows(scores, ids):
    return np.lexsort((ids, -scores))


def at_capacity(frame, scores, capacity, cutoff=threshold):
    ids = frame.property_id.to_numpy(dtype=np.int64)
    elevated = frame.ratio_next.to_numpy() > cutoff
    volume = np.maximum(frame.intensity_next - cutoff * frame.expected_next, 0).to_numpy() * frame.area_next.to_numpy()
    k = ceil(capacity * len(frame))
    metrics = []
    for column in scores.T:
        chosen = ordered_rows(column, ids)[:k]
        hits = int(elevated[chosen].sum())
        metrics.append({'slots': k, 'hits': hits, 'precision': hits / k,
                        'recall': hits / elevated.sum(), 'volumeCapture': float(volume[chosen].sum() / volume.sum())})
    return metrics


validation_results = at_capacity(validation_pairs, rank_scores(validation_pairs), .10)
default_policy = int(np.argmax([result['precision'] for result in validation_results]))
print(pd.DataFrame({'Policy': policies, 'Validation precision @10%': [f"{r['precision']:.1%}" for r in validation_results]}).to_string(index=False))
print('Demo default frozen on validation:', policies[default_policy])

                Policy Validation precision @10%
    Reported intensity                     65.7%
   Peer-adjusted ratio                     74.8%
Gallons above expected                     53.3%
        Predicted risk                     74.8%
Demo default frozen on validation: Peer-adjusted ratio


## Measure the review yield on 2023-to-2024 transitions

Candidate records with missing or ineligible 2024 follow-up are excluded from the scored ranking. The comparison below quantifies how those candidates differ from those with usable follow-up. Review outcomes are evaluated among the observed pairs.

In [6]:
future_screened, _, _ = screen(future)
test_2023 = history.loc[history.report_year.eq('2023') & history.split.eq('test')]
test_2024 = future_screened.loc[future_screened.split.eq('test')]
test_pairs = transitions(test_2023, test_2024)
assert set(test_pairs.group).isdisjoint(training_pairs.group)
assert set(test_pairs.group).isdisjoint(validation_pairs.group)
assert test_pairs.property_id.is_unique

test_start = candidate_features(test_2023)
linked = test_start.property_id.isin(test_pairs.property_id)
elevated = test_pairs.ratio_next.to_numpy() > threshold
high_now = test_pairs.ratio.to_numpy() > threshold
print(f'2023 candidates: {len(test_start):,}; eligible 2024 follow-up: {len(test_pairs):,} ({linked.mean():.1%}) in {test_pairs.group.nunique():,} groups')
for name, frame in [('With follow-up', test_start.loc[linked]), ('Without follow-up', test_start.loc[~linked])]:
    print(f'{name}: {len(frame):,}; 2023 median intensity {frame.intensity.median():.2f}; 2023 elevated {frame.ratio.gt(threshold).mean():.1%}')
print(f'2024 elevated: {elevated.sum()} / {len(elevated)} ({elevated.mean():.1%})')
print(f'2024 elevated if already high in 2023: {elevated[high_now].mean():.1%}; if below cutoff in 2023: {elevated[~high_now].mean():.1%}')

scores = rank_scores(test_pairs)
test_results = [at_capacity(test_pairs, scores, capacity) for capacity in capacities]
for capacity, metrics in zip(capacities, test_results):
    table = pd.DataFrame(metrics, index=policies)
    for column in ['precision', 'recall', 'volumeCapture']:
        table[column] = table[column].map('{:.1%}'.format)
    print(f'\nReview capacity {capacity:.0%}')
    print(table.to_string())

2023 candidates: 2,647; eligible 2024 follow-up: 1,843 (69.6%) in 1,747 groups
With follow-up: 1,843; 2023 median intensity 53.35; 2023 elevated 10.9%
Without follow-up: 804; 2023 median intensity 50.93; 2023 elevated 10.3%
2024 elevated: 197 / 1843 (10.7%)
2024 elevated if already high in 2023: 79.1%; if below cutoff in 2023: 2.3%

Review capacity 5%
                        slots  hits precision recall volumeCapture
Reported intensity         93    83     89.2%  42.1%         65.2%
Peer-adjusted ratio        93    89     95.7%  45.2%         84.2%
Gallons above expected     93    66     71.0%  33.5%         85.3%
Predicted risk             93    89     95.7%  45.2%         84.6%

Review capacity 10%
                        slots  hits precision recall volumeCapture
Reported intensity        185   133     71.9%  67.5%         80.5%
Peer-adjusted ratio       185   153     82.7%  77.7%         91.9%
Gallons above expected    185   125     67.6%  63.5%         94.7%
Predicted risk        

## Check uncertainty and whether results depend on the cutoff

In [7]:
indices_by_group = list(test_pairs.groupby('group').indices.values())
ids = test_pairs.property_id.to_numpy(dtype=np.int64)
rng = np.random.default_rng(663)
differences = np.empty((1000, len(policies) - 1))
for iteration in range(len(differences)):
    chosen_groups = rng.integers(len(indices_by_group), size=len(indices_by_group))
    indices = np.concatenate([indices_by_group[index] for index in chosen_groups])
    k = ceil(.10 * len(indices))
    sampled_precision = []
    for policy_scores in scores.T:
        selected = ordered_rows(policy_scores[indices], ids[indices])[:k]
        sampled_precision.append(float(elevated[indices[selected]].mean()))
    differences[iteration] = np.asarray(sampled_precision[1:]) - sampled_precision[0]

intervals = np.quantile(differences, [.025, .975], axis=0).T
primary = test_results[1]
for name, result, (lo, hi) in zip(policies[1:], primary[1:], intervals):
    gap = result['precision'] - primary[0]['precision']
    print(f'{name}: {100*gap:+.1f} percentage points vs intensity, 95% interval [{100*lo:+.1f}, {100*hi:+.1f}]')

for quantile, cutoff in zip([85, 95], alternative_thresholds[[0, 2]]):
    sensitivity = at_capacity(test_pairs, scores, .10, float(cutoff))
    values = {name: f"{r['precision']:.1%}" for name, r in zip(policies, sensitivity)}
    print(f'{quantile}th percentile target ({cutoff:.2f}×): {values}')

Peer-adjusted ratio: +10.8 percentage points vs intensity, 95% interval [+6.0, +15.2]
Gallons above expected: -4.3 percentage points vs intensity, 95% interval [-12.4, +2.7]
Predicted risk: +10.8 percentage points vs intensity, 95% interval [+5.9, +15.2]
85th percentile target (1.99×): {'Reported intensity': '84.9%', 'Peer-adjusted ratio': '92.4%', 'Gallons above expected': '78.9%', 'Predicted risk': '92.4%'}
95th percentile target (2.70×): {'Reported intensity': '48.1%', 'Peer-adjusted ratio': '55.7%', 'Gallons above expected': '44.9%', 'Predicted risk': '55.7%'}


## Related work and scope for the term paper

EPA Portfolio Manager already benchmarks multifamily water use. Kontokosta and Jain studied determinants of water use using NYC multifamily disclosures. Wang and El-Gohary studied building-level water consumption prediction. The contribution here is the comparison of explicit queue policies under the same number of reviews and the separation of two outcomes: finding properties with elevated reports and capturing gallons above the benchmark threshold. [EPA Water Score](https://www.energystar.gov/buildings/resources-audience/multifamily-housing/water-score) · [Kontokosta and Jain](https://www.sciencedirect.com/science/article/abs/pii/S2210670715000633) · [Wang and El-Gohary](https://experts.illinois.edu/en/publications/machine-learning-based-prediction-of-building-water-consumption-f/)

Three substantive work packages for the team:

1. **Data engineering and measurement.** Reproduce cohort screening and property linkage. Explain the incomplete follow-up and manually review ambiguous property records. Document the CSV contract and quality checks.
2. **Decision experiment.** Reproduce the paired evaluation, justify the proxy outcome and the audit budgets, and explain the trade-off between case yield and volume capture. Present the sensitivity analysis and group-level uncertainty.
3. **Tool and workflow.** Demonstrate the same portfolio under two ranking rules at equal capacity, show how poor records enter a separate data-quality queue, and export a shortlist a manager can work through. Validate the workflow on representative cases and document the audit actions.

The 2024 benchmark was evaluated earlier in this project. These ranking results are a retrospective extension with fixed score rules, a pre-specified outcome and disjoint property groups. Confirmatory evidence would require another reporting year or independent audit records. The study has no verified repair, leak or water-saving labels.

## Paper argument and interpretation

**Proposed title.** Who Gets Audited First? Budget-Constrained Water Review for Multifamily Buildings.

At 10% capacity, the peer-adjusted queue found 153 of the 197 properties that reported elevated use the next year, compared with 133 from the same 185 reviews ranked by raw intensity. The precision difference was +10.8 percentage points, with a 95% group-bootstrap interval of +6.0 to +15.2. The trained risk score selected the same number of elevated properties. Ranking by gallons above expected captured 94.7% of the observed future gallons above the elevated-use threshold, compared with 91.9% for the peer ratio and 80.5% for intensity, while finding fewer elevated properties. The risk score had the highest volume capture at 10% capacity in this comparison (95.2%). These volume shares are descriptive. The analysis estimates no savings from intervention.

The main outcome is defined relative to the same benchmark used in the peer ratio. This makes the study a test of how well policies anticipate next-year benchmark exceedance. Among linked properties, 79.1% of those above the cutoff in 2023 were also above it in 2024. That persistence helps explain why an adjusted ratio was competitive with the learned risk score. The same direction of difference against raw intensity held when using the 85th and 95th percentile cutoffs.

The strongest limitation is follow-up. Only 1,843 of 2,647 eligible 2023 properties had a usable matched 2024 report. The remaining properties had a lower median starting intensity (50.93 versus 53.35 gal/ft²/year). This difference and the absence of audit outcomes limit generalization. For a viva, show the raw-intensity and peer-ratio shortlists at the same budget, then switch the evidence chart from case yield to gallons to explain why the ranking depends on the manager's goal.